In [5]:
## SETUP 

from pathlib import Path
import numpy as np
import pandas as pd
import mne
from specparam import SpectralGroupModel

psg_dir   = Path("/Users/elizabethkaplan/Desktop/MASS_PSG")
data_dir  = Path("/Users/elizabethkaplan/Desktop/SS2_Data")
save_dir  = Path("/Users/elizabethkaplan/Desktop/SS2_Results/time_resolved")
save_dir.mkdir(exist_ok=True)

EXCLUDE   = {"01-02-0019"}

subject_ids = [f"01-02-{i:04d}" for i in range(1, 20)]
subject_ids = [s for s in subject_ids if s not in EXCLUDE]

# Analysis parameters
win_sec    = 1.0
step_sec   = 0.25
fmin, fmax = 1.0, 45.0
r2_thresh  = 0.85

stage_order = {
    "Sleep stage W": 0, "Sleep stage 1": 1, "Sleep stage 2": 2,
    "Sleep stage 3": 3, "Sleep stage 4": 3,
    "Sleep stage R": 4, "Sleep stage ?": np.nan,
}

print(f"Subjects to process: {len(subject_ids)}")

Subjects to process: 18


In [ ]:
## CELL — Multi-subject loop with checkpointing

all_subjects = {}
failed       = []

for subj_id in subject_ids:

    # ── Checkpoint: skip if already processed ─────────────────────────────
    save_path = save_dir / f"{subj_id}_time_resolved.npz"
    if save_path.exists():
        print(f"[SKIP] {subj_id}: already processed, loading from disk")
        d = np.load(save_path, allow_pickle=True)
        all_subjects[subj_id] = {k: d[k] for k in d.files}
        # numpy saves object arrays differently — fix string arrays
        all_subjects[subj_id]["window_stages"] = list(all_subjects[subj_id]["window_stages"])
        continue

    psg_path     = psg_dir  / f"{subj_id} PSG.edf"
    kc_path      = data_dir / f"{subj_id} KComplexes_E1.edf"
    spindle_path = data_dir / f"{subj_id} Spindles_E1.edf"
    staging_path = data_dir / f"{subj_id} Base.edf"

    if not all(p.exists() for p in [psg_path, kc_path, spindle_path, staging_path]):
        print(f"[SKIP] {subj_id}: missing file")
        failed.append((subj_id, "missing file"))
        continue

    try:
        print(f"\nProcessing {subj_id}...")

        raw      = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
        annot    = mne.read_annotations(kc_path)
        spindles = mne.read_annotations(spindle_path)
        stages   = mne.read_annotations(staging_path)
        sfreq    = raw.info["sfreq"]

        c3_idx = [i for i, ch in enumerate(raw.ch_names) if "C3" in ch.upper()]
        if not c3_idx:
            raise ValueError(f"No C3 channel. Available: {raw.ch_names}")
        c3_data = raw.get_data(picks=c3_idx)

        kc_onsets      = np.array([a["onset"] for a in annot])
        spindle_onsets = np.array([a["onset"] for a in spindles])

        nperseg  = int(win_sec  * sfreq)
        step_smp = int(step_sec * sfreq)
        n_samp   = c3_data.shape[1]
        starts   = np.arange(0, n_samp - nperseg + 1, step_smp)

        windows = np.stack([c3_data[0, s:s + nperseg].astype(float) for s in starts])

        psds_mt, freqs_tf = mne.time_frequency.psd_array_multitaper(
            windows, sfreq=sfreq, fmin=fmin, fmax=fmax,
            bandwidth=4.0, verbose=False,
        )

        t_centers = (starts + nperseg / 2) / sfreq
        t_hours   = t_centers / 3600.0

        window_stages = []
        for t in t_centers:
            label = "Sleep stage ?"
            for ann in stages:
                if ann["onset"] <= t < ann["onset"] + ann["duration"]:
                    label = ann["description"].strip()
                    break
            window_stages.append(label)
        stage_numeric = np.array([stage_order.get(s, np.nan) for s in window_stages])

        fg = SpectralGroupModel(
            peak_width_limits=[1, 12], max_n_peaks=8,
            min_peak_height=0.0, peak_threshold=2.0,
            aperiodic_mode="fixed", verbose=False,
        )
        fg.fit(freqs_tf, np.clip(psds_mt, 1e-30, None), freq_range=[1, 45])

        exponent = np.array([r.aperiodic_fit[1] for r in fg.results.group_results])
        offset   = np.array([r.aperiodic_fit[0] for r in fg.results.group_results])
        r2       = np.array([r.metrics.get("gof_rsquared", np.nan) for r in fg.results.group_results])

        good           = r2 >= r2_thresh
        exponent_clean = np.where(good, exponent, np.nan)
        offset_clean   = np.where(good, offset,   np.nan)

        subj_data = {
            "t_centers":      t_centers,
            "t_hours":        t_hours,
            "exponent":       exponent,
            "offset":         offset,
            "exponent_clean": exponent_clean,
            "offset_clean":   offset_clean,
            "r2":             r2,
            "good":           good,
            "window_stages":  np.array(window_stages),
            "stage_numeric":  stage_numeric,
            "kc_onsets":      kc_onsets,
            "spindle_onsets": spindle_onsets,
            "sfreq":          np.array(sfreq),
            "freqs_tf":       freqs_tf,
        }

        # ── Save immediately ──────────────────────────────────────────────
        np.savez(save_path, **subj_data)
        print(f"  Saved to {save_path.name}")

        all_subjects[subj_id] = subj_data

        print(f"  {subj_id}: {len(starts)} windows, "
              f"good={good.sum()} ({100*good.mean():.1f}%), "
              f"KCs={len(kc_onsets)}, spindles={len(spindle_onsets)}")

    except Exception as e:
        import traceback
        print(f"[FAIL] {subj_id}: {e}")
        traceback.print_exc()
        failed.append((subj_id, str(e)))

print(f"\nDone: {len(all_subjects)} subjects, {len(failed)} failed")

[SKIP] 01-02-0001: already processed, loading from disk
[SKIP] 01-02-0002: already processed, loading from disk

Processing 01-02-0003...


/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encount

  Saved to 01-02-0003_time_resolved.npz
  01-02-0003: 147037 windows, good=132272 (90.0%), KCs=533, spindles=143

Processing 01-02-0004...


/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


  Saved to 01-02-0004_time_resolved.npz
  01-02-0004: 112013 windows, good=110143 (98.3%), KCs=607, spindles=253

Processing 01-02-0005...


/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/elizabethkaplan/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


In [ ]:
## CELL — Stage-level aperiodic averages across subjects  (#1)

stage_names = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
rows = []

for subj_id, d in all_subjects.items():
    for stage_val, stage_name in stage_names.items():
        mask = (d["stage_numeric"] == stage_val) & d["good"]
        if mask.sum() < 10:
            continue
        rows.append({
            "subject":      subj_id,
            "stage":        stage_name,
            "mean_exponent": np.nanmean(d["exponent_clean"][mask]),
            "std_exponent":  np.nanstd(d["exponent_clean"][mask]),
            "mean_offset":   np.nanmean(d["offset_clean"][mask]),
            "std_offset":    np.nanstd(d["offset_clean"][mask]),
            "n_windows":     mask.sum(),
        })

stage_df = pd.DataFrame(rows)
print(stage_df.groupby("stage")[["mean_exponent", "mean_offset"]].describe().round(3))

In [ ]:
## CELL — Event-locked exponent averaging around KCs and spindles  (#2)

pre_sec  = 30.0   # seconds before event
post_sec = 30.0   # seconds after event
step_sec_val = 0.25

n_bins   = int((pre_sec + post_sec) / step_sec_val)
t_event  = np.linspace(-pre_sec, post_sec, n_bins)

kc_epochs      = []
spindle_epochs = []

for subj_id, d in all_subjects.items():
    t_c  = d["t_centres"]
    exp  = d["exponent_clean"]

    for onset in d["kc_onsets"]:
        # find window indices within ±30s of this KC
        idx = np.where((t_c >= onset - pre_sec) & (t_c <= onset + post_sec))[0]
        if len(idx) != n_bins:
            continue
        kc_epochs.append(exp[idx])

    for onset in d["spindle_onsets"]:
        idx = np.where((t_c >= onset - pre_sec) & (t_c <= onset + post_sec))[0]
        if len(idx) != n_bins:
            continue
        spindle_epochs.append(exp[idx])

kc_epochs      = np.array(kc_epochs)
spindle_epochs = np.array(spindle_epochs)

print(f"KC epochs:      {kc_epochs.shape}")
print(f"Spindle epochs: {spindle_epochs.shape}")

In [ ]:
## CELL — Plot event-locked exponent  (#2 continued)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, epochs, label, color in zip(
    axes,
    [kc_epochs, spindle_epochs],
    ["K-Complex", "Spindle"],
    ["#2c3e8c", "#e07b54"],
):
    mean = np.nanmean(epochs, axis=0)
    sem  = np.nanstd(epochs, axis=0) / np.sqrt(np.sum(~np.isnan(epochs), axis=0))

    ax.plot(t_event, mean, color=color, linewidth=2)
    ax.fill_between(t_event, mean - sem, mean + sem, alpha=0.3, color=color)
    ax.axvline(0, color="k", linestyle="--", linewidth=1, label="Event onset")
    ax.set_xlabel("Time relative to event (s)", fontsize=12)
    ax.set_ylabel("Aperiodic exponent", fontsize=12)
    ax.set_title(f"{label} (n={len(epochs)})", fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.2)

plt.suptitle("Event-locked aperiodic exponent (group, ±30s)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─── SAVE OUTPUTS ────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd

#save_dir = out_dir # adjust out_dir to wherever you want


for subj_id, d in all_subjects.items():

    # ── 1. Window-level time series (main output) ──────────────────────────
    win_df = pd.DataFrame({
        "t_center":     d["t_center"],
        "t_center_hr":    d["t_hours"],
        "stage":          d["window_stages"],
        "stage_numeric":  d["stage_numeric"],
        "exponent":       d["exponent"],
        "offset":         d["offset"],
        "exponent_clean": d["exponent_clean"],
        "offset_clean":   d["offset_clean"],
        "r2":             d["r2"],
        "good":           d["good"],
    })
    win_df.to_csv(save_dir / f"{subj_id}_windows.csv", index=False)

    # ── 2. KC and spindle onsets ───────────────────────────────────────────
    pd.DataFrame({"kc_onset_s": d["kc_onsets"]}).to_csv(
        save_dir / f"{subj_id}_kc_onsets.csv", index=False
    )
    pd.DataFrame({"spindle_onset_s": d["spindle_onsets"]}).to_csv(
        save_dir / f"{subj_id}_spindle_onsets.csv", index=False
    )

# ── 3. Shared frequency axis (same for all subjects) ──────────────────────
first = next(iter(all_subjects.values()))
pd.DataFrame({"freq_hz": first["freqs_tf"]}).to_csv(
    save_dir / "freqs.csv", index=False
)

# ── 4. Cross-subject summary (one row per subject) ────────────────────────
rows = []
for subj_id, d in all_subjects.items():
    good = d["good"]
    rows.append({
        "subj_id":           subj_id,
        "n_windows":         len(d["t_centres"]),
        "n_good_windows":    int(good.sum()),
        "pct_good":          round(100 * good.mean(), 1),
        "mean_exponent":     np.nanmean(d["exponent_clean"]),
        "mean_offset":       np.nanmean(d["offset_clean"]),
        "mean_r2":           np.nanmean(d["r2"]),
        "n_kc":              len(d["kc_onsets"]),
        "n_spindles":        len(d["spindle_onsets"]),
    })
pd.DataFrame(rows).to_csv(save_dir / "summary_all_subjects.csv", index=False)

print(f"Saved {len(all_subjects)} subjects to {save_dir}")

In [ ]:
import pickle

save_dir = Path("/Users/elizabethkaplan/Desktop/SS2_Results/time_resolved")
save_dir.mkdir(parents=True, exist_ok=True)

with open(save_dir / "all_subjects.pkl", "wb") as f:
    pickle.dump(all_subjects, f)

print("Saved.")